# [0.6] How to Know When an Interpretability Result Is Fake - Solutions

## Core Question

Can a compact GT-0 audit reject leakage, cherry-picking, probe overfit, and weak steering claims before later notebooks trust real-model evidence?

## Learning Objectives

- Validate each fake-result detector against default and alternate fixtures.
- Interpret the negative-result signature table.
- Keep CUDA evidence scoped to the methodology claim.

### Exercise - solution verification

> Difficulty: medium  
> Importance: high

Reference validation notebook for the GT-0 fake-result diagnostics section.


<details>
<summary>Help - interpreting this notebook</summary>

The successful outcome is mostly negative: the synthetic claims are designed to be bogus, and the diagnostics should reject them. Do not reinterpret these numbers as a real-model mechanism.

</details>

In [1]:
import json
import sys
from pathlib import Path

chapter = "chapter0_fundamentals"
section = "part6_fake_interpretability_results"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_fake_interpretability_results.tests as tests
from chapter0_fundamentals.exercises.part6_fake_interpretability_results import solutions


## Sub-function Tests

<details>
<summary>Expected output</summary>

Every test prints its `All tests ... passed!` line, including the alternate-fixture controls.

</details>

In [2]:
tests.test_binary_accuracy_thresholds_signed_scores(solutions.binary_accuracy)
tests.test_label_leakage_report_flags_direct_label_feature(solutions.label_leakage_report)
tests.test_label_leakage_report_uses_supplied_feature_indices(solutions.label_leakage_report)
tests.test_cherry_pick_report_compares_selected_to_population(solutions.cherry_pick_report)
tests.test_cherry_pick_report_rejects_representative_selection(solutions.cherry_pick_report)
tests.test_probe_overfit_report_requires_heldout_gap(solutions.probe_overfit_report)
tests.test_probe_overfit_report_rejects_generalizing_probe(solutions.probe_overfit_report)
tests.test_random_direction_control_report_rejects_weak_claim(solutions.random_direction_control_report)
tests.test_random_direction_control_report_accepts_strong_claim(solutions.random_direction_control_report)
tests.test_fake_result_audit_report_aggregates_all_failure_modes(solutions.fake_result_audit_report)
tests.test_notebook_contract(solutions.run_smoke_test)


All tests in `test_binary_accuracy_thresholds_signed_scores` passed!
All tests in `test_label_leakage_report_flags_direct_label_feature` passed!
All tests in `test_label_leakage_report_uses_supplied_feature_indices` passed!
All tests in `test_cherry_pick_report_compares_selected_to_population` passed!
All tests in `test_cherry_pick_report_rejects_representative_selection` passed!
All tests in `test_probe_overfit_report_requires_heldout_gap` passed!
All tests in `test_probe_overfit_report_rejects_generalizing_probe` passed!
All tests in `test_random_direction_control_report_rejects_weak_claim` passed!
All tests in `test_random_direction_control_report_accepts_strong_claim` passed!
All tests in `test_fake_result_audit_report_aggregates_all_failure_modes` passed!
All tests in `test_notebook_contract` passed!


## Smoke-Test Contract

In [3]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["contract_passed"]
assert contract["leakage"]["leaked_feature_accuracy"] == 1.0
assert contract["leakage"]["shifted_no_leak_accuracy"] == 0.0
assert contract["cherry_pick"]["inflation_ratio"] >= 3.0
assert contract["probe_overfit"]["train_accuracy"] == 1.0
assert contract["probe_overfit"]["heldout_accuracy"] == 0.5
assert contract["random_direction"]["detects_random_direction_failure"]
contract


{'leakage': {'leaked_feature_index': 0,
  'leaked_feature_accuracy': 1.0,
  'shifted_no_leak_accuracy': 0.0,
  'accuracy_gap': 1.0,
  'detects_leakage': True},
 'cherry_pick': {'selected_mean_effect': 1.5499998331069946,
  'population_mean_effect': 0.3700000047683716,
  'population_median_effect': 0.09000000357627869,
  'inflation_ratio': 4.189188684138882,
  'detects_cherry_picking': True},
 'probe_overfit': {'train_accuracy': 1.0,
  'heldout_accuracy': 0.5,
  'generalization_gap': 0.5,
  'detects_overfit': True},
 'random_direction': {'claimed_effect': 0.0500024579312309,
  'random_p95_effect': 0.1183513992303592,
  'effect_gap': -0.0683489412991283,
  'passes_random_control': False,
  'detects_random_direction_failure': True},
 'audit': {'leakage_detected': True,
  'cherry_pick_detected': True,
  'probe_overfit_detected': True,
  'random_direction_failure_detected': True,
  'all_bogus_results_flagged': True},
 'contract_passed': True}

## Signature Result

![Fake interpretability signature result](expected_outputs/fake_interpretability_signature_result.svg)

<details>
<summary>Interpreting this result</summary>

The report succeeds because all four injected bogus claims are rejected. This is a prerequisite for later claims, not evidence for any particular model circuit.

</details>

In [4]:
tests.test_committed_gpu_report_records_fake_result_signature()
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["leaked_feature_accuracy"] == 1.0
assert gpu["shifted_no_leak_accuracy"] == 0.0
assert gpu["leakage_gap"] >= 0.5
assert gpu["cherry_pick_inflation"] >= 3.0
assert gpu["selected_mean_effect"] >= 5 * gpu["population_median_effect"]
assert gpu["probe_train_accuracy"] == 1.0
assert gpu["probe_heldout_accuracy"] == 0.5
assert gpu["probe_overfit_gap"] >= 0.35
assert gpu["random_direction_control_rejects_claim"]
assert gpu["random_direction_effect_gap"] < 0.25
assert gpu["all_bogus_results_flagged"]
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "device",
    "leaked_feature_accuracy",
    "shifted_no_leak_accuracy",
    "cherry_pick_inflation",
    "probe_overfit_gap",
    "random_direction_effect_gap",
    "peak_vram_gb",
]}


All tests in `test_committed_gpu_report_records_fake_result_signature` passed!


{'device': 'NVIDIA GeForce RTX 5090 Laptop GPU',
 'leaked_feature_accuracy': 1.0,
 'shifted_no_leak_accuracy': 0.0,
 'cherry_pick_inflation': 4.1891886689003375,
 'probe_overfit_gap': 0.5,
 'random_direction_effect_gap': -0.0683489412991283,
 'peak_vram_gb': 0.031256675720214844}

## Limitations

These are GT-0 generated tensors. The section validates the diagnostic habit, not a real-model mechanism.

## Bonus - Anomaly Hunting

Try lowering each failure signal until the detector should issue a warning rather than a hard rejection.